open balanced dataset and start set up tokenizer

In [41]:
import pandas as pd
import torch
from datasets import load_dataset,ClassLabel, Dataset

from transformers import BertForSequenceClassification, BertTokenizer, BertConfig

In [42]:
dataset = load_dataset("csv", data_files="archive/balancedSubset.csv")

In [43]:
genres = list(set(dataset["train"]["genre"]))
class_label = ClassLabel(names=genres)
dataset = dataset["train"].cast_column("genre", class_label)

In [44]:
dataset_split : Dataset = dataset.train_test_split(test_size=0.2, seed=69, stratify_by_column="genre")

In [45]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

LITTLE SETUP

In [46]:
def tokenize_function (set : Dataset) :
    return tokenizer(set['lyrics'], padding="max_length", truncation=True,max_length=512)

In [47]:
tokenized_dataset = dataset_split.map(tokenize_function, batched=True)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 6291.52 examples/s]


In [48]:
from transformers import AutoModelForSequenceClassification
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

print(model.config)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2044.61it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.2.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

